Subject: Basic test of data preprocessing pipeline

Body: Extracted sub-items from JSON files and merged them into single .pkl and .csv files, saved to the current directory.

NOTE: This is a test version, please do not use it yet.

TODO:

Currently using hardcoded local absolute paths (will be updated).

Currently outputting to the local working directory (will be updated).

Usability of the output data is currently unverified.

In [1]:
# -*- coding: utf-8 -*-
"""
Evolutionary Heuristic Design Data Pipeline - Final Aligned Version
Features:
1. Auto-discovery via os.walk (Your advantage)
2. JSON-level exception check (Teammate alignment - Critical for BinPacking)
3. Strict Audit Reporting
"""

import os
import json
import pandas as pd

# --- AUDIT LOG ---
audit_stats = {
    "1_total_files_found": 0,
    "2_excluded_filename_exception": 0,  # 檔名含 _Exception
    "3_excluded_json_inner_exception": 0, # JSON 內含 "exception": true (關鍵差異點)
    "4_excluded_json_error": 0,
    "5_excluded_missing_offspring": 0,
    "6_dropped_nan_objective": 0,
    "7_dropped_empty_code": 0,
    "8_dropped_duplicate_id": 0,
    "FINAL_DATASET_COUNT": 0
}

# 任務分組統計
task_audit = {}

def fast_parse_strategy(filename):
    """從檔名解析策略"""
    parts = filename.split('_')
    try:
        if 'op' in parts:
            idx = parts.index('op')
            return parts[idx + 1]
    except (ValueError, IndexError):
        pass
    return "unknown"

def update_task_stat(task_name, stat_key):
    """更新特定任務的計數"""
    if task_name not in task_audit:
        task_audit[task_name] = {'raw': 0, 'kept': 0, 'diff_json_exception': 0}
    task_audit[task_name][stat_key] += 1

def normalize_task_name(raw_app_type, instance_scale):
    """標準化任務名稱"""
    name_map = {
        'bin_greedy': 'BinPacking',
        'cvrp_lns': 'CVRP',
        'premarshalling_astar': 'Premarshalling',
        'puzzle_astar': 'SlidingPuzzle'
    }
    # 嘗試匹配
    base_name = name_map.get(raw_app_type, raw_app_type)
    if base_name == 'HotAI Material' or base_name == '':
        for key, standard_name in name_map.items():
            if key in instance_scale:
                return standard_name
        return instance_scale 
    return base_name

def scan_all_heuristic_datasets(root_path):
    """
    階段一：掃描與提取
    新增：檢查 JSON 內部的 'exception' 欄位，這應該能解決 BinPacking 的差異。
    """
    all_data = []
    print(f"Starting directory scan at: {root_path}")
    
    for root, dirs, files in os.walk(root_path):
        if os.path.basename(root) == 'all_programs':
            path_parts = root.replace('\\', '/').split('/')
            app_type = path_parts[-4] if len(path_parts) >= 4 else "Unknown"
            instance_scale = path_parts[-3] if len(path_parts) >= 3 else "Unknown"
            
            # 預判 Task Name (僅用於審計)
            current_task = normalize_task_name(app_type, instance_scale)
            
            for filename in os.listdir(root):
                if filename.endswith('.json'):
                    audit_stats["1_total_files_found"] += 1
                    update_task_stat(current_task, 'raw')
                    
                    # Filter 1: 檔名 Exception
                    if '_Exception' in filename:
                        audit_stats["2_excluded_filename_exception"] += 1
                        continue
                    
                    file_path = os.path.join(root, filename)
                    try:
                        with open(file_path, 'r', encoding='utf-8') as f:
                            content = json.load(f)
                            
                            # Filter 2: JSON 內部 Exception (!!! 這是關鍵修正 !!!)
                            # 組員代碼邏輯: if obj.get("exception"): return None
                            if content.get("exception"):
                                audit_stats["3_excluded_json_inner_exception"] += 1
                                update_task_stat(current_task, 'diff_json_exception') # 記錄這是哪個任務掉的
                                continue

                            offspring = content.get('offspring', None)
                            if not isinstance(offspring, dict): 
                                audit_stats["5_excluded_missing_offspring"] += 1
                                continue
                            
                            # 提取數據
                            algo_data = offspring.get('algorithm', [""])
                            algo = algo_data[0] if isinstance(algo_data, list) else algo_data
                            
                            code_data = offspring.get('code', [])
                            code = "".join(code_data) if isinstance(code_data, list) else code_data
                            
                            obj = offspring.get('objective', None)
                            
                            # ID 獲取 (與組員邏輯對齊：offspring_id -> id -> filename)
                            h_id = offspring.get('offspring_id') or offspring.get('id') or filename
                            
                            all_data.append({
                                'heuristic_id': str(h_id),
                                'raw_app_type': app_type,
                                'instance_scale': instance_scale,
                                'filename': filename,
                                'strategy': fast_parse_strategy(filename),
                                'algorithm': algo.strip('{} '),
                                'code': code,
                                'objective': obj
                            })
                    except (json.JSONDecodeError, OSError):
                        audit_stats["4_excluded_json_error"] += 1
                        continue
                        
    return pd.DataFrame(all_data)

def clean_and_dedup(df):
    """階段二：清洗與去重"""
    if df.empty: return df
    df = df.copy()
    
    # 1. Objective 必須有效
    df['objective'] = pd.to_numeric(df['objective'], errors='coerce')
    len_before = len(df)
    df = df.dropna(subset=['objective'])
    audit_stats["6_dropped_nan_objective"] += (len_before - len(df))
    
    # 2. Code 必須非空 (strip後)
    len_before = len(df)
    df = df[df['code'].str.strip() != ""]
    audit_stats["7_dropped_empty_code"] += (len_before - len(df))
    
    # 3. ID 去重 (保留第一個)
    len_before = len(df)
    df = df.drop_duplicates(subset=['heuristic_id'], keep='first')
    audit_stats["8_dropped_duplicate_id"] += (len_before - len(df))
    
    return df

def finalize_task_name(row):
    return normalize_task_name(row['raw_app_type'], row['instance_scale'])

# --- EXECUTION ---
# Root path to your dataset
root_material = r"F:\KIT\HotAI\18151101\HotAI Material"

# 1. Ingestion
df_raw = scan_all_heuristic_datasets(root_material)

if not df_raw.empty:
    # 2. Cleaning
    df_final = clean_and_dedup(df_raw)
    
    # 3. Standardization
    df_final['task_name'] = df_final.apply(finalize_task_name, axis=1)
    
    # 4. Timeout Tagging
    df_final['is_timeout'] = False
    timeout_threshold = 10000 
    df_final.loc[
        (df_final['task_name'].str.contains('Puzzle|Premarshalling', case=False)) & 
        (df_final['objective'] > timeout_threshold), 
        'is_timeout'
    ] = True

    audit_stats["FINAL_DATASET_COUNT"] = len(df_final)

    # 更新每個 Task 的最終數量
    for task in df_final['task_name'].unique():
        if task in task_audit:
            task_audit[task]['kept'] = len(df_final[df_final['task_name'] == task])

    # 5. Persistence
    pickle_path = "all_heuristics_dataset.pkl"
    csv_path = "all_heuristics_dataset.csv"
    
    df_final.to_pickle(pickle_path)
    df_final.to_csv(csv_path, index=False)
    
    # 6. Detailed Report
    print("\n" + "="*70)
    print(f"{'TASK NAME':<20} | {'RAW FILES':<10} | {'INTERNAL EXC':<12} | {'FINAL KEPT':<10}")
    print("-" * 70)
    
    for task, stats in task_audit.items():
        # INTERNAL EXC 是指 JSON 內部有 exception: true 但檔名正常的
        print(f"{task:<20} | {stats['raw']:<10} | {stats['diff_json_exception']:<12} | {stats['kept']:<10}")
        
    print("-" * 70)
    print("\n[Audit Breakdown]")
    print(f"1. Total Found       : {audit_stats['1_total_files_found']}")
    print(f"2. Filename _Exc     : -{audit_stats['2_excluded_filename_exception']}")
    print(f"3. JSON Inner _Exc   : -{audit_stats['3_excluded_json_inner_exception']} (Potential Diff Source)")
    print(f"4. Missing Offspring : -{audit_stats['5_excluded_missing_offspring']}")
    print(f"5. Duplicate IDs     : -{audit_stats['8_dropped_duplicate_id']}")
    print("="*70)
    print(f"Final Count       : {len(df_final)}")
    print(f"Saved to          : {os.path.abspath(pickle_path)}")

else:
    print("Error: No data found.")

Starting directory scan at: F:\KIT\HotAI\18151101\HotAI Material

TASK NAME            | RAW FILES  | INTERNAL EXC | FINAL KEPT
----------------------------------------------------------------------
BinPacking           | 4847       | 0            | 4445      
CVRP                 | 1640       | 0            | 1557      
Premarshalling       | 4920       | 0            | 4647      
SlidingPuzzle        | 4920       | 0            | 4920      
----------------------------------------------------------------------

[Audit Breakdown]
1. Total Found       : 16327
2. Filename _Exc     : -758
3. JSON Inner _Exc   : -0 (Potential Diff Source)
4. Missing Offspring : -0
5. Duplicate IDs     : -0
Final Count       : 15569
Saved to          : f:\KIT\HotAI\Task\ppp-performance-prediction\notebooks\Yunshi\all_heuristics_dataset.pkl
